# 03. Movie Metadata Unification

이 노트북의 목적은 `Movie_Master_v1.csv`를 기준 테이블로 두고 Wavve 크롤링 메타데이터와 KOBIS 보완 메타데이터를 통합하는 것이다.

이 단계의 최종 산출물은 이후 `05_content_feature_engineering.ipynb`에서 사용할 영화 단위 통합 메타데이터 테이블이다.

핵심 원칙은 다음과 같다.

1. 기준 테이블은 반드시 `Movie_Master_v1.csv`이다.
2. Wavve 메타데이터는 OTT 서비스 맥락의 1차 메타데이터로 본다.
3. KOBIS 메타데이터는 Wavve 미수집분을 보완하는 2차 메타데이터로 본다.
4. KOBIS는 이미 필터링된 999행 파일만 존재하므로 원 후보군 중 재선택은 불가능하다.
5. 따라서 KOBIS는 제목 유사도와 제목 내 연도 힌트로 신뢰도를 점검한다.
6. 저신뢰 KOBIS 매칭은 CSV에는 남기되, `use_for_content_features=0`으로 두어 콘텐츠 피처 생성에서는 제외한다.

출력 파일은 `v1`과 `v2` 두 종류다.

- `movie_metadata_unified_v1.csv`: 느슨한 통합본. Wavve 우선, KOBIS 보완을 그대로 사용한다.
- `movie_metadata_unified_v2.csv`: 분석용 권장본. KOBIS 저신뢰 매칭을 제외한다.

In [1]:
from pathlib import Path
import json
import re
from difflib import SequenceMatcher
from collections import Counter

import numpy as np
import pandas as pd

In [2]:
def find_project_root(start: Path | None = None) -> Path:
    """Find project root. Works from notebook folder or copied standalone package."""
    if start is None:
        start = Path.cwd()
    start = start.resolve()
    candidates = [start, *start.parents]
    for p in candidates:
        if (p / '_data').exists() and (p / 'notebooks').exists():
            return p
        if p.name == 'park.ingyeom':
            return p
    # Fallback for execution checks in a temporary directory.
    return start

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / '_data'
RAW_DIR = DATA_DIR / '01_raw'
INTERIM_DIR = DATA_DIR / '02_interim'
PROCESSED_DIR = DATA_DIR / '03_processed'
REPORTS_DIR = PROJECT_ROOT / 'reports'
TABLES_DIR = REPORTS_DIR / 'tables'

for d in [RAW_DIR, INTERIM_DIR, PROCESSED_DIR, TABLES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('PROJECT_ROOT:', PROJECT_ROOT)
print('RAW_DIR:', RAW_DIR)
print('INTERIM_DIR:', INTERIM_DIR)
print('TABLES_DIR:', TABLES_DIR)

PROJECT_ROOT: /mnt/data/test_repo_03/park.ingyeom
RAW_DIR: /mnt/data/test_repo_03/park.ingyeom/_data/01_raw
INTERIM_DIR: /mnt/data/test_repo_03/park.ingyeom/_data/02_interim
TABLES_DIR: /mnt/data/test_repo_03/park.ingyeom/reports/tables


In [3]:
FILE_NAMES = {
    'movie_master': 'Movie_Master_v1.csv',
    'wavve': 'wavve_movies_filtered_by  정규식 (1)(1).csv',
    'kobis': 'wavve_notfound_kobis_filtered_by_char_match(1).csv',
    'view': 'View_History_v1.csv',
}

def resolve_input_file(file_name: str) -> Path:
    candidates = [
        RAW_DIR / file_name,
        DATA_DIR / file_name,
        INTERIM_DIR / file_name,
        PROJECT_ROOT / file_name,
        Path('/mnt/data') / file_name,
    ]
    for p in candidates:
        if p.exists():
            return p
    raise FileNotFoundError(f'Cannot find input file: {file_name}')

PATH_MOVIE = resolve_input_file(FILE_NAMES['movie_master'])
PATH_WAVVE = resolve_input_file(FILE_NAMES['wavve'])
PATH_KOBIS = resolve_input_file(FILE_NAMES['kobis'])
PATH_VIEW = resolve_input_file(FILE_NAMES['view'])

PATH_MOVIE, PATH_WAVVE, PATH_KOBIS, PATH_VIEW

(PosixPath('/mnt/data/test_repo_03/park.ingyeom/_data/01_raw/Movie_Master_v1.csv'),
 PosixPath('/mnt/data/test_repo_03/park.ingyeom/_data/01_raw/wavve_movies_filtered_by  정규식 (1)(1).csv'),
 PosixPath('/mnt/data/test_repo_03/park.ingyeom/_data/01_raw/wavve_notfound_kobis_filtered_by_char_match(1).csv'),
 PosixPath('/mnt/data/test_repo_03/park.ingyeom/_data/01_raw/View_History_v1.csv'))

In [4]:
movie = pd.read_csv(PATH_MOVIE)
wavve = pd.read_csv(PATH_WAVVE)
kobis = pd.read_csv(PATH_KOBIS)
view = pd.read_csv(PATH_VIEW)

file_summary = pd.DataFrame([
    {'name': 'Movie_Master', 'path': str(PATH_MOVIE), 'rows': len(movie), 'cols': movie.shape[1]},
    {'name': 'Wavve', 'path': str(PATH_WAVVE), 'rows': len(wavve), 'cols': wavve.shape[1]},
    {'name': 'KOBIS', 'path': str(PATH_KOBIS), 'rows': len(kobis), 'cols': kobis.shape[1]},
    {'name': 'View_History', 'path': str(PATH_VIEW), 'rows': len(view), 'cols': view.shape[1]},
])
file_summary.to_csv(TABLES_DIR / '03_movie_metadata_input_file_summary.csv', index=False, encoding='utf-8-sig')
file_summary

,name,path,rows,cols
0,Movie_Master,/mnt/data/test_repo_03/park.ingyeom/_data/01_r...,14018,3
1,Wavve,/mnt/data/test_repo_03/park.ingyeom/_data/01_r...,4060,31
2,KOBIS,/mnt/data/test_repo_03/park.ingyeom/_data/01_r...,999,13
3,View_History,/mnt/data/test_repo_03/park.ingyeom/_data/01_r...,106205,5


In [5]:
GENRE_MAP = {
    '드라마': '드라마',
    '액션': '액션',
    '스릴러': '스릴러/범죄',
    '범죄': '스릴러/범죄',
    '느와르': '스릴러/범죄',
    '미스터리': '스릴러/범죄',
    '코미디': '코미디',
    '로맨스': '로맨스',
    '멜로/로맨스': '로맨스',
    'SF': 'SF/판타지',
    '판타지': 'SF/판타지',
    'SF/판타지': 'SF/판타지',
    '모험': '모험/어드벤처',
    '어드벤처': '모험/어드벤처',
    '애니메이션': '애니메이션/키즈',
    '키즈': '애니메이션/키즈',
    '공포': '공포',
    '공포(호러)': '공포',
    '호러': '공포',
    '다큐멘터리': '다큐/교양',
    '교양': '다큐/교양',
    '가족': '가족',
    '전쟁': '전쟁/재난',
    '재난': '전쟁/재난',
    '전쟁/재난': '전쟁/재난',
    '음악': '기타',
    '공연': '기타',
    '스포츠': '기타',
    '서부': '기타',
    '서부극(웨스턴)': '기타',
    '사극': '기타',
    '무협': '기타',
    '성인물(에로)': '기타',
    '에로티시즘': '기타',
    '극장판': '기타',
    '단편': '기타',
    '뮤지컬': '기타',
    '기타': '기타',
}

GENRE_ORDER = [
    '드라마', '액션', '스릴러/범죄', '코미디', '로맨스', 'SF/판타지',
    '모험/어드벤처', '애니메이션/키즈', '공포', '다큐/교양', '가족',
    '전쟁/재난', '기타',
]

COUNTRY_MAP = {
    '대한민국': '한국',
    '한국': '한국',
    '미국': '미국',
    '일본': '일본',
    '중국': '중국',
    '홍콩': '홍콩',
    '영국': '영국',
    '프랑스': '프랑스',
    '독일': '독일',
    '캐나다': '캐나다',
    '러시아': '러시아',
    '대만': '대만',
    '호주': '호주',
    '이탈리아': '이탈리아',
    '스페인': '스페인',
    '기타': '기타',
}

In [6]:
def normalize_title(value) -> str:
    if pd.isna(value):
        return ''
    s = str(value).strip().lower()
    s = re.sub(r'[\[\]{}<>〈〉《》『』「」【】]', '', s)
    s = s.replace('（', '(').replace('）', ')')
    s = re.sub(r'\s+', '', s)
    s = re.sub(r"[-_:;,.!?'\"`~·ㆍ/\\|+*&^%$#@=]", '', s)
    return s

def clean_title_for_similarity(value) -> str:
    if pd.isna(value):
        return ''
    s = str(value)
    s = re.sub(r'[\(\[\{（][^\)\]\}）]*[\)\]\}）]', '', s)
    return s.strip()

def normalize_title_no_year(value) -> str:
    return normalize_title(clean_title_for_similarity(value))

def extract_year_hint(value) -> str:
    if pd.isna(value):
        return ''
    s = str(value)
    m = re.search(r'[\(\[\{（]\s*((?:19|20)\d{2})\s*[\)\]\}）]', s)
    return m.group(1) if m else ''

def sequence_similarity(a, b) -> float:
    a_norm = normalize_title_no_year(a)
    b_norm = normalize_title_no_year(b)
    if not a_norm or not b_norm:
        return 0.0
    return SequenceMatcher(None, a_norm, b_norm).ratio()

def split_multi(value) -> list[str]:
    if pd.isna(value):
        return []
    s = str(value).strip()
    if not s or s.lower() == 'nan':
        return []
    return [p.strip() for p in re.split(r'[|,;/]', s) if p and p.strip()]

def unique_join(values) -> str:
    seen = []
    for value in values:
        if pd.isna(value):
            continue
        s = str(value).strip()
        if not s or s.lower() == 'nan':
            continue
        if s not in seen:
            seen.append(s)
    return '|'.join(seen)

def normalize_genres(raw_values) -> list[str]:
    out = set()
    for raw in raw_values:
        for part in split_multi(raw):
            mapped = GENRE_MAP.get(part, GENRE_MAP.get(part.replace(' ', ''), '기타'))
            out.add(mapped)
    return [g for g in GENRE_ORDER if g in out]

def normalize_countries(raw_values) -> list[str]:
    out = []
    for raw in raw_values:
        for part in split_multi(raw):
            mapped = COUNTRY_MAP.get(part, part)
            if mapped not in out:
                out.append(mapped)
    return out

def normalize_age_rating_from_wavve(value) -> str:
    if pd.isna(value):
        return ''
    s = str(value).strip()
    if not s or s.lower() == 'nan':
        return ''
    if s.endswith('.0'):
        s = s[:-2]
    if s in {'0', '전체', '전체관람가'}:
        return '전체'
    if s in {'7', '7세'}:
        return '7세'
    if s in {'12', '12세', '12세관람가', '12세이상관람가'}:
        return '12세'
    if s in {'15', '15세', '15세관람가', '15세이상관람가'}:
        return '15세'
    if s in {'18', '19', '청소년관람불가', '18세관람가'}:
        return '청불'
    return s

def normalize_age_rating_from_kobis(value) -> str:
    if pd.isna(value):
        return ''
    s = str(value).strip()
    if not s or s.lower() == 'nan':
        return ''
    if '전체' in s or '모든 관람객' in s or '연소자관람가' in s:
        return '전체'
    if '12' in s:
        return '12세'
    if '15' in s or '중학생' in s:
        return '15세'
    if '청소년관람불가' in s or '18' in s or '연소자관람불가' in s or '미성년자관람불가' in s or '고등학생' in s:
        return '청불'
    return s

def open_year_from_open_dt(value) -> str:
    if pd.isna(value):
        return ''
    s = str(value).strip()
    digits = re.sub(r'\D', '', s)
    if len(digits) >= 4 and digits[:4].startswith(('19', '20')):
        return digits[:4]
    return ''

def parse_float(value):
    if pd.isna(value):
        return np.nan
    try:
        return float(value)
    except Exception:
        return np.nan

def parse_int(value):
    f = parse_float(value)
    if pd.isna(f):
        return np.nan
    return int(round(f))

def median_or_blank(values) -> str:
    vals = [v for v in values if not pd.isna(v)]
    if not vals:
        return ''
    return f'{float(np.median(vals)):.1f}'

def bool_int(condition) -> int:
    return int(bool(condition))

In [7]:
movie = movie.copy()
wavve = wavve.copy()
kobis = kobis.copy()

movie['title_key_norm'] = movie['movie_title'].map(normalize_title)
wavve['title_key_norm'] = wavve['query_title'].map(normalize_title)
kobis['title_key_norm'] = kobis['wavve_title'].map(normalize_title)

key_summary = pd.DataFrame([
    {'table': 'Movie_Master', 'rows': len(movie), 'unique_title_key': movie['title_key_norm'].nunique(), 'duplicate_title_key_rows': int(movie['title_key_norm'].duplicated(keep=False).sum())},
    {'table': 'Wavve', 'rows': len(wavve), 'unique_title_key': wavve['title_key_norm'].nunique(), 'duplicate_title_key_rows': int(wavve['title_key_norm'].duplicated(keep=False).sum())},
    {'table': 'KOBIS', 'rows': len(kobis), 'unique_title_key': kobis['title_key_norm'].nunique(), 'duplicate_title_key_rows': int(kobis['title_key_norm'].duplicated(keep=False).sum())},
])
key_summary.to_csv(TABLES_DIR / '03_title_key_summary.csv', index=False, encoding='utf-8-sig')
key_summary

,table,rows,unique_title_key,duplicate_title_key_rows
0,Movie_Master,14018,14012,12
1,Wavve,4060,3576,864
2,KOBIS,999,999,0


In [8]:
def aggregate_wavve(wavve_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for key, g in wavve_df.groupby('title_key_norm', dropna=False):
        if not key:
            continue
        genres_raw = []
        tags_raw = []
        countries_raw = []
        ages_norm = []
        runtimes = []
        years = []
        for _, r in g.iterrows():
            genres_raw.extend(split_multi(r.get('genre')))
            tags_raw.extend(split_multi(r.get('tags')))
            countries_raw.extend(split_multi(r.get('country')))
            age = normalize_age_rating_from_wavve(r.get('targetage'))
            if age:
                ages_norm.append(age)
            sec = parse_float(r.get('playtime_sec'))
            if not pd.isna(sec) and sec > 0:
                runtimes.append(sec / 60.0)
            year = parse_int(r.get('originalreleaseyear'))
            if not pd.isna(year):
                years.append(int(year))
        rows.append({
            'title_key_norm': key,
            'wavve_match_count': len(g),
            'wavve_movieids': unique_join(g.get('movieid', pd.Series(dtype=object)).tolist()),
            'wavve_titles': unique_join(g.get('title', pd.Series(dtype=object)).tolist()),
            'wavve_genre_raw': unique_join(genres_raw),
            'wavve_tags_raw': unique_join(tags_raw),
            'wavve_country_raw': unique_join(countries_raw),
            'wavve_age_rating_norm': unique_join(ages_norm),
            'wavve_runtime_min': median_or_blank(runtimes),
            'wavve_release_year': str(int(round(np.median(years)))) if years else '',
        })
    return pd.DataFrame(rows)

wavve_agg = aggregate_wavve(wavve)
wavve_agg.head()

,title_key_norm,wavve_match_count,wavve_movieids,wavve_titles,wavve_genre_raw,wavve_tags_raw,wavve_country_raw,wavve_age_rating_norm,wavve_runtime_min,wavve_release_year
0,007북경특급2,1,MV_AN01_AN0000000020,007 북경특급2,액션|드라마,액션|드라마,홍콩,15세,87.4,2014
1,100일동안100가지로100퍼센트행복찾기,1,MV_CV01_KE0000012213,100일 동안 100가지로 100퍼센트 행복찾기,코미디,코미디,독일,15세,111.0,2019
2,101마리의달마시안개(1961),1,MV_CA01_DY0000011204,(더빙) 101마리의 달마시안 개,애니메이션|모험,애니메이션|모험,미국,전체,79.3,1961
3,108영웅전설의무공,1,MV_ST01_ST000000839,108영웅: 전설의 무공,액션|무협,액션|무협,중국,15세,91.2,2017
4,10년,1,MV_CK01_TCO000012364,10년,드라마,드라마,일본,전체,99.3,2019


In [9]:
def aggregate_kobis(kobis_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for key, g in kobis_df.groupby('title_key_norm', dropna=False):
        if not key:
            continue
        genres_raw = []
        countries_raw = []
        ages_norm = []
        runtimes = []
        prdt_years = []
        open_years = []
        for _, r in g.iterrows():
            genres_raw.extend(split_multi(r.get('genreNm')))
            countries_raw.extend(split_multi(r.get('nationNm')))
            age = normalize_age_rating_from_kobis(r.get('watchGrade'))
            if age:
                ages_norm.append(age)
            runtime = parse_float(r.get('showTm'))
            if not pd.isna(runtime) and runtime > 0:
                runtimes.append(runtime)
            y = parse_int(r.get('prdtYear'))
            if not pd.isna(y):
                prdt_years.append(int(y))
            oy = open_year_from_open_dt(r.get('openDt'))
            if oy:
                open_years.append(oy)
        first = g.iloc[0]
        first_runtime = parse_float(first.get('showTm'))
        first_year = parse_int(first.get('prdtYear'))
        rows.append({
            'title_key_norm': key,
            'kobis_match_count': len(g),
            'kobis_movie_codes': unique_join(g.get('MovieCd', pd.Series(dtype=object)).tolist()),
            'kobis_titles': unique_join(g.get('movieNm(api)', pd.Series(dtype=object)).tolist()),
            'kobis_titles_en': unique_join(g.get('movieNmEn', pd.Series(dtype=object)).tolist()),
            'kobis_type_names': unique_join(g.get('typeNm', pd.Series(dtype=object)).tolist()),
            'kobis_directors': unique_join(g.get('directors', pd.Series(dtype=object)).tolist()),
            'kobis_actors': unique_join(g.get('actors', pd.Series(dtype=object)).tolist()),
            'kobis_wavve_titles': unique_join(g.get('wavve_title', pd.Series(dtype=object)).tolist()),
            'kobis_watch_grade_raw': unique_join(g.get('watchGrade', pd.Series(dtype=object)).tolist()),
            'kobis_genre_raw': unique_join(genres_raw),
            'kobis_country_raw': unique_join(countries_raw),
            'kobis_age_rating_norm': unique_join(ages_norm),
            'kobis_runtime_min': median_or_blank(runtimes),
            'kobis_release_year': str(int(round(np.median(prdt_years)))) if prdt_years else '',
            'kobis_open_year': unique_join(open_years),
            'kobis_selected_movie_code': first.get('MovieCd', ''),
            'kobis_selected_title': first.get('movieNm(api)', ''),
            'kobis_selected_title_en': first.get('movieNmEn', ''),
            'kobis_selected_type': first.get('typeNm', ''),
            'kobis_selected_directors': first.get('directors', ''),
            'kobis_selected_actors': first.get('actors', ''),
            'kobis_selected_watch_grade_raw': first.get('watchGrade', ''),
            'kobis_selected_genre_raw': first.get('genreNm', ''),
            'kobis_selected_country_raw': first.get('nationNm', ''),
            'kobis_selected_age_rating_norm': normalize_age_rating_from_kobis(first.get('watchGrade')),
            'kobis_selected_runtime_min': f'{float(first_runtime):.1f}' if not pd.isna(first_runtime) else '',
            'kobis_selected_release_year': str(int(first_year)) if not pd.isna(first_year) else '',
            'kobis_selected_open_year': open_year_from_open_dt(first.get('openDt')),
        })
    return pd.DataFrame(rows)

kobis_agg = aggregate_kobis(kobis)
kobis_agg.head()

,title_key_norm,kobis_match_count,kobis_movie_codes,kobis_titles,kobis_titles_en,kobis_type_names,kobis_directors,kobis_actors,kobis_wavve_titles,kobis_watch_grade_raw,kobis_genre_raw,kobis_country_raw,kobis_age_rating_norm,kobis_runtime_min,kobis_release_year,kobis_open_year,kobis_selected_movie_code,kobis_selected_title,kobis_selected_title_en,kobis_selected_type,kobis_selected_directors,kobis_selected_actors,kobis_selected_watch_grade_raw,kobis_selected_genre_raw,kobis_selected_country_raw,kobis_selected_age_rating_norm,kobis_selected_runtime_min,kobis_selected_release_year,kobis_selected_open_year
0,007스카이폴,1,20113461,007 스카이폴,SKYFALL,장편,샘 멘데스,다니엘 크레이그|하비에르 바르뎀|주디 덴치|랄프 파인즈|나오미 해리스,007스카이폴,15세이상관람가,액션,미국|영국,15세,143.0,2011,2012,20113461,007 스카이폴,SKYFALL,장편,샘 멘데스,다니엘 크레이그|하비에르 바르뎀|주디 덴치|랄프 파인즈|나오미 해리스,15세이상관람가,액션,미국|영국,15세,143.0,2011,2012
1,007스펙터,1,20157432,007 스펙터,Spectre,장편,샘 멘데스,다니엘 크레이그|레아 세이두|크리스토프 왈츠|모니카 벨루치,007스펙터,15세이상관람가,액션|어드벤처|범죄|스릴러,영국|미국,15세,147.0,2015,2015,20157432,007 스펙터,Spectre,장편,샘 멘데스,다니엘 크레이그|레아 세이두|크리스토프 왈츠|모니카 벨루치,15세이상관람가,액션|어드벤처|범죄|스릴러,영국|미국,15세,147.0,2015,2015
2,007제로,1,2022A107,007 제로,Double zero,온라인전용,,,007제로,,액션,프랑스,,,2004,,2022A107,007 제로,Double zero,온라인전용,NaN,NaN,NaN,액션,프랑스,,,2004,
3,10미니츠곤,1,20198121,10 미니츠 곤,10 MINUTES GONE,장편,브라이언 A 밀러,브루스 윌리스|마이클 치클리스,10미니츠곤,15세이상관람가,액션|범죄,캐나다|미국,15세,95.0,2019,2019,20198121,10 미니츠 곤,10 MINUTES GONE,장편,브라이언 A 밀러,브루스 윌리스|마이클 치클리스,15세이상관람가,액션|범죄,캐나다|미국,15세,95.0,2019,2019
4,12디재스터,1,20142008,12 디재스터,12 Disasters,기타,스티븐 R. 몬로,에드 퀸|마그다 아파노위즈,12디재스터,,SF,미국|캐나다,,,2012,,20142008,12 디재스터,12 Disasters,기타,스티븐 R. 몬로,에드 퀸|마그다 아파노위즈,NaN,SF,미국|캐나다,,,2012,


In [10]:
def assess_kobis_quality(movie_title, row: pd.Series) -> dict:
    if row is None or row.empty:
        return {
            'title_year_hint': '',
            'kobis_title_similarity': '',
            'kobis_year_diff_min': '',
            'kobis_year_match_flag': '',
            'kobis_low_confidence_reasons': '',
            'kobis_quality_flag': '',
            'use_kobis_for_content_features': 0,
        }
    selected_title = row.get('kobis_selected_title', '') or row.get('kobis_titles', '')
    sim = sequence_similarity(movie_title, selected_title)
    year_hint = extract_year_hint(movie_title) or extract_year_hint(row.get('kobis_wavve_titles', ''))
    candidate_years = []
    for field in ['kobis_selected_release_year', 'kobis_selected_open_year', 'kobis_release_year', 'kobis_open_year']:
        for part in split_multi(row.get(field, '')):
            y = parse_int(part)
            if not pd.isna(y) and 1880 <= int(y) <= 2035:
                candidate_years.append(int(y))
    year_diff_min = ''
    year_match_flag = ''
    if year_hint:
        yh = int(year_hint)
        if candidate_years:
            diff = min(abs(yh - y) for y in candidate_years)
            year_diff_min = str(diff)
            year_match_flag = '1' if diff <= 1 else '0'
        else:
            year_match_flag = '0'
    reasons = []
    if sim < 0.60:
        reasons.append('low_title_similarity')
    if year_hint and year_match_flag == '0':
        reasons.append('year_hint_mismatch')
    if not str(row.get('kobis_selected_genre_raw', '')).strip():
        reasons.append('missing_genre')
    if not str(row.get('kobis_selected_country_raw', '')).strip():
        reasons.append('missing_country')
    if not str(row.get('kobis_selected_age_rating_norm', '')).strip():
        reasons.append('missing_age_rating')
    try:
        if int(row.get('kobis_match_count', 0)) > 1:
            reasons.append('multiple_kobis_rows_for_same_title')
    except Exception:
        pass
    if 'low_title_similarity' in reasons or 'year_hint_mismatch' in reasons or 'multiple_kobis_rows_for_same_title' in reasons:
        quality = 'D_kobis_low_confidence_excluded'
        use = 0
    elif year_hint:
        quality = 'B_kobis_high_confidence'
        use = 1
    else:
        quality = 'C_kobis_medium_confidence'
        use = 1 if sim >= 0.80 else 0
        if use == 0:
            quality = 'D_kobis_low_confidence_excluded'
            if 'medium_title_similarity_without_year_hint' not in reasons:
                reasons.append('medium_title_similarity_without_year_hint')
    return {
        'title_year_hint': year_hint,
        'kobis_title_similarity': f'{sim:.4f}' if selected_title else '',
        'kobis_year_diff_min': year_diff_min,
        'kobis_year_match_flag': year_match_flag,
        'kobis_low_confidence_reasons': '|'.join(reasons),
        'kobis_quality_flag': quality,
        'use_kobis_for_content_features': use,
    }

def make_content_flags(unified_genres, unified_countries, unified_age, unified_runtime, unified_release_year, wavve_tags_raw):
    runtime = parse_float(unified_runtime)
    year = parse_int(unified_release_year)
    is_kids_animation = ('애니메이션/키즈' in unified_genres) or ('키즈' in split_multi(wavve_tags_raw))
    is_family_content = ('가족' in unified_genres) or is_kids_animation or unified_age in {'전체', '7세', '12세'}
    return {
        'is_kids_animation': bool_int(is_kids_animation),
        'is_family_content': bool_int(is_family_content),
        'is_adult_content': bool_int(unified_age == '청불'),
        'is_korean_content': bool_int('한국' in unified_countries or '대한민국' in unified_countries),
        'is_us_content': bool_int('미국' in unified_countries),
        'is_japanese_content': bool_int('일본' in unified_countries),
        'is_recent_content': bool_int((not pd.isna(year)) and int(year) >= 2018),
        'is_old_content': bool_int((not pd.isna(year)) and int(year) <= 2010),
        'is_long_movie': bool_int((not pd.isna(runtime)) and runtime >= 120),
        'is_short_content': bool_int((not pd.isna(runtime)) and runtime < 60),
    }

In [11]:
base = movie.merge(wavve_agg, on='title_key_norm', how='left').merge(kobis_agg, on='title_key_norm', how='left')

# Fill selected audit columns as strings for stable downstream processing.
for col in base.columns:
    if col not in ['MOVIE_NUM', 'ott_release_month']:
        base[col] = base[col].fillna('')

base['source_wavve_flag'] = (base['wavve_match_count'].astype(str).str.len() > 0).astype(int)
base['source_kobis_flag'] = (base['kobis_match_count'].astype(str).str.len() > 0).astype(int)
base['metadata_covered_flag'] = ((base['source_wavve_flag'] == 1) | (base['source_kobis_flag'] == 1)).astype(int)
base['metadata_source'] = np.select(
    [
        (base['source_wavve_flag'] == 1) & (base['source_kobis_flag'] == 1),
        base['source_wavve_flag'] == 1,
        base['source_kobis_flag'] == 1,
    ],
    ['both', 'wavve', 'kobis'],
    default='missing',
)

quality_rows = []
for _, r in base.iterrows():
    quality_rows.append(assess_kobis_quality(r['movie_title'], r))
quality_df = pd.DataFrame(quality_rows)
base = pd.concat([base.reset_index(drop=True), quality_df.reset_index(drop=True)], axis=1)

base[['MOVIE_NUM', 'movie_title', 'metadata_source', 'source_wavve_flag', 'source_kobis_flag', 'metadata_covered_flag']].head()

,MOVIE_NUM,movie_title,metadata_source,source_wavve_flag,source_kobis_flag,metadata_covered_flag
0,0,걸어서하늘까지(1992),missing,0,0,0
1,1,너와극장에서,missing,0,0,0
2,2,가려진시간[가치봄],missing,0,0,0
3,3,그링고,wavve,1,0,1
4,4,스위치(2010),missing,0,0,0


In [12]:
def build_unified_version(base_df: pd.DataFrame, version: str) -> pd.DataFrame:
    rows = []
    for _, r in base_df.iterrows():
        has_w = int(r['source_wavve_flag']) == 1
        has_k = int(r['source_kobis_flag']) == 1
        if version == 'v1':
            use_for_content = 1 if (has_w or has_k) else 0
            metadata_quality = 'A_wavve' if has_w else ('B_or_C_kobis_unchecked' if has_k else 'E_missing')
            use_kobis_values = has_k and not has_w
        elif version == 'v2':
            if has_w:
                use_for_content = 1
                metadata_quality = 'A_wavve'
                use_kobis_values = False
            elif has_k and int(r['use_kobis_for_content_features']) == 1:
                use_for_content = 1
                metadata_quality = r['kobis_quality_flag']
                use_kobis_values = True
            elif has_k:
                use_for_content = 0
                metadata_quality = r['kobis_quality_flag'] or 'D_kobis_low_confidence_excluded'
                use_kobis_values = False
            else:
                use_for_content = 0
                metadata_quality = 'E_missing'
                use_kobis_values = False
        else:
            raise ValueError("version must be 'v1' or 'v2'")
        if has_w:
            raw_genre_values = [r.get('wavve_genre_raw', '')]
            raw_country_values = [r.get('wavve_country_raw', '')]
            unified_age = r.get('wavve_age_rating_norm', '') or r.get('kobis_age_rating_norm', '')
            unified_runtime = r.get('wavve_runtime_min', '') or r.get('kobis_runtime_min', '')
            unified_release_year = r.get('wavve_release_year', '') or r.get('kobis_release_year', '')
        elif use_kobis_values:
            raw_genre_values = [r.get('kobis_selected_genre_raw', '') or r.get('kobis_genre_raw', '')]
            raw_country_values = [r.get('kobis_selected_country_raw', '') or r.get('kobis_country_raw', '')]
            unified_age = r.get('kobis_selected_age_rating_norm', '') or r.get('kobis_age_rating_norm', '')
            unified_runtime = r.get('kobis_selected_runtime_min', '') or r.get('kobis_runtime_min', '')
            unified_release_year = r.get('kobis_selected_release_year', '') or r.get('kobis_release_year', '')
        else:
            raw_genre_values = []
            raw_country_values = []
            unified_age = ''
            unified_runtime = ''
            unified_release_year = ''
        unified_genres = normalize_genres(raw_genre_values)
        unified_countries = normalize_countries(raw_country_values)
        flags = make_content_flags(unified_genres, unified_countries, unified_age, unified_runtime, unified_release_year, r.get('wavve_tags_raw', ''))
        row = {
            'MOVIE_NUM': r['MOVIE_NUM'],
            'movie_title': r['movie_title'],
            'ott_release_month': r.get('ott_release_month', ''),
            'title_key_norm': r.get('title_key_norm', ''),
            'metadata_source': r.get('metadata_source', ''),
            'metadata_quality': metadata_quality,
            'use_for_content_features': use_for_content,
            'source_wavve_flag': int(r['source_wavve_flag']),
            'source_kobis_flag': int(r['source_kobis_flag']),
            'metadata_covered_flag': int(r['metadata_covered_flag']),
            'wavve_match_count': r.get('wavve_match_count', ''),
            'kobis_match_count': r.get('kobis_match_count', ''),
            'wavve_movieids': r.get('wavve_movieids', ''),
            'kobis_movie_codes': r.get('kobis_movie_codes', ''),
            'wavve_titles': r.get('wavve_titles', ''),
            'kobis_titles': r.get('kobis_titles', ''),
            'kobis_titles_en': r.get('kobis_titles_en', ''),
            'kobis_type_names': r.get('kobis_type_names', ''),
            'kobis_selected_movie_code': r.get('kobis_selected_movie_code', ''),
            'kobis_selected_title': r.get('kobis_selected_title', ''),
            'kobis_selected_title_en': r.get('kobis_selected_title_en', ''),
            'kobis_selected_type': r.get('kobis_selected_type', ''),
            'kobis_selected_directors': r.get('kobis_selected_directors', ''),
            'kobis_selected_actors': r.get('kobis_selected_actors', ''),
            'title_year_hint': r.get('title_year_hint', ''),
            'kobis_title_similarity': r.get('kobis_title_similarity', ''),
            'kobis_year_diff_min': r.get('kobis_year_diff_min', ''),
            'kobis_year_match_flag': r.get('kobis_year_match_flag', ''),
            'kobis_low_confidence_reasons': r.get('kobis_low_confidence_reasons', ''),
            'wavve_genre_raw': r.get('wavve_genre_raw', ''),
            'kobis_genre_raw': r.get('kobis_genre_raw', ''),
            'kobis_selected_genre_raw': r.get('kobis_selected_genre_raw', ''),
            'unified_genres': '|'.join(unified_genres),
            'wavve_tags_raw': r.get('wavve_tags_raw', ''),
            'wavve_country_raw': r.get('wavve_country_raw', ''),
            'kobis_country_raw': r.get('kobis_country_raw', ''),
            'kobis_selected_country_raw': r.get('kobis_selected_country_raw', ''),
            'unified_countries': '|'.join(unified_countries),
            'wavve_age_rating_norm': r.get('wavve_age_rating_norm', ''),
            'kobis_age_rating_norm': r.get('kobis_age_rating_norm', ''),
            'kobis_selected_age_rating_norm': r.get('kobis_selected_age_rating_norm', ''),
            'unified_age_rating': unified_age,
            'wavve_runtime_min': r.get('wavve_runtime_min', ''),
            'kobis_runtime_min': r.get('kobis_runtime_min', ''),
            'kobis_selected_runtime_min': r.get('kobis_selected_runtime_min', ''),
            'unified_runtime_min': unified_runtime,
            'wavve_release_year': r.get('wavve_release_year', ''),
            'kobis_release_year': r.get('kobis_release_year', ''),
            'kobis_open_year': r.get('kobis_open_year', ''),
            'kobis_selected_release_year': r.get('kobis_selected_release_year', ''),
            'kobis_selected_open_year': r.get('kobis_selected_open_year', ''),
            'unified_release_year': unified_release_year,
            **flags,
        }
        rows.append(row)
    return pd.DataFrame(rows)

unified_v1 = build_unified_version(base, 'v1')
unified_v2 = build_unified_version(base, 'v2')

unified_v1.shape, unified_v2.shape

((14018, 62), (14018, 62))

In [13]:
def coverage_by_scope(unified_df: pd.DataFrame, version: str) -> pd.DataFrame:
    view_movie_nums = set(view['MOVIE_NUM'].dropna().astype(int).unique())
    u = unified_df.copy()
    u['MOVIE_NUM_int'] = u['MOVIE_NUM'].astype(int)
    all_row = {
        'version': version,
        'scope': 'Movie_Master_all',
        'total_movies': len(u),
        'metadata_covered': int(u['metadata_covered_flag'].sum()),
        'usable_for_content_features': int(u['use_for_content_features'].sum()),
        'metadata_coverage_rate': float(u['metadata_covered_flag'].mean()),
        'usable_rate': float(u['use_for_content_features'].mean()),
    }
    uv = u[u['MOVIE_NUM_int'].isin(view_movie_nums)].copy()
    view_row = {
        'version': version,
        'scope': 'View_History_movies_only',
        'total_movies': len(uv),
        'metadata_covered': int(uv['metadata_covered_flag'].sum()),
        'usable_for_content_features': int(uv['use_for_content_features'].sum()),
        'metadata_coverage_rate': float(uv['metadata_covered_flag'].mean()),
        'usable_rate': float(uv['use_for_content_features'].mean()),
    }
    return pd.DataFrame([all_row, view_row])

coverage = pd.concat([
    coverage_by_scope(unified_v1, 'v1'),
    coverage_by_scope(unified_v2, 'v2'),
], ignore_index=True)
coverage.to_csv(TABLES_DIR / '03_metadata_coverage_summary.csv', index=False, encoding='utf-8-sig')
coverage

,version,scope,total_movies,metadata_covered,usable_for_content_features,metadata_coverage_rate,usable_rate
0,v1,Movie_Master_all,14018,4578,4578,0.326580,0.326580
1,v1,View_History_movies_only,5196,4576,4576,0.880677,0.880677
2,v2,Movie_Master_all,14018,4578,4551,0.326580,0.324654
3,v2,View_History_movies_only,5196,4576,4549,0.880677,0.875481


In [14]:
quality_counts = pd.concat([
    unified_v1.assign(version='v1').groupby(['version', 'metadata_source', 'metadata_quality', 'use_for_content_features']).size().reset_index(name='count'),
    unified_v2.assign(version='v2').groupby(['version', 'metadata_source', 'metadata_quality', 'use_for_content_features']).size().reset_index(name='count'),
], ignore_index=True)
quality_counts.to_csv(TABLES_DIR / '03_metadata_quality_counts.csv', index=False, encoding='utf-8-sig')
quality_counts

,version,metadata_source,metadata_quality,use_for_content_features,count
0,v1,kobis,B_or_C_kobis_unchecked,1,1000
1,v1,missing,E_missing,0,9440
2,v1,wavve,A_wavve,1,3578
3,v2,kobis,B_kobis_high_confidence,1,121
4,v2,kobis,C_kobis_medium_confidence,1,852
5,v2,kobis,D_kobis_low_confidence_excluded,0,27
6,v2,missing,E_missing,0,9440
7,v2,wavve,A_wavve,1,3578


In [15]:
low_confidence_kobis = unified_v2.loc[
    unified_v2['metadata_quality'].astype(str).str.startswith('D_'),
    [
        'MOVIE_NUM', 'movie_title', 'metadata_source', 'metadata_quality',
        'kobis_selected_title', 'title_year_hint', 'kobis_selected_release_year',
        'kobis_selected_open_year', 'kobis_title_similarity', 'kobis_low_confidence_reasons',
        'kobis_selected_genre_raw', 'kobis_selected_country_raw', 'kobis_selected_age_rating_norm'
    ]
].copy()
low_confidence_kobis.to_csv(TABLES_DIR / '03_low_confidence_kobis_rows.csv', index=False, encoding='utf-8-sig')
low_confidence_kobis.head(20)

,MOVIE_NUM,movie_title,metadata_source,metadata_quality,kobis_selected_title,title_year_hint,kobis_selected_release_year,kobis_selected_open_year,kobis_title_similarity,kobis_low_confidence_reasons,kobis_selected_genre_raw,kobis_selected_country_raw,kobis_selected_age_rating_norm
1282,1334,제인에어(1996),kobis,D_kobis_low_confidence_excluded,제인 에어,1996,1944,,1.0000,year_hint_mismatch|missing_age_rating,드라마,미국,
2809,2923,오션스일레븐(2001),kobis,D_kobis_low_confidence_excluded,오션스 일레븐,2001,1960,,1.0000,year_hint_mismatch|missing_age_rating,범죄|코미디,미국,
3091,3213,변신(2019)[가치봄],kobis,D_kobis_low_confidence_excluded,변신,2019,2023,,1.0000,year_hint_mismatch|missing_age_rating,드라마,한국,
4365,4526,[OCEAN공개]커넥트(2020),kobis,D_kobis_low_confidence_excluded,커넥트,2020,2022,,1.0000,year_hint_mismatch|missing_age_rating,드라마,이탈리아,
4392,4553,운동회(2018),kobis,D_kobis_low_confidence_excluded,운동회,2018,2002,,1.0000,year_hint_mismatch|missing_genre|missing_age_r...,,한국,
4848,5031,눈길(2017),kobis,D_kobis_low_confidence_excluded,눈길,2017,2025,,1.0000,year_hint_mismatch|missing_age_rating,드라마,한국,
5710,5928,강적(2006),kobis,D_kobis_low_confidence_excluded,강적,2006,2023,,1.0000,year_hint_mismatch|missing_age_rating,드라마,한국,
5783,6003,침입자(2020)[별도편성][예고편],kobis,D_kobis_low_confidence_excluded,침입자,2020,2004,,1.0000,year_hint_mismatch|missing_age_rating,드라마,프랑스,
5810,6031,당신이잠든사이에(1995),kobis,D_kobis_low_confidence_excluded,당신이 잠든 사이에,1995,2008,2008,1.0000,year_hint_mismatch,코미디,한국,15세
6274,6515,래빗홀(2010),kobis,D_kobis_low_confidence_excluded,래빗홀,2010,2021,,1.0000,year_hint_mismatch|missing_age_rating,애니메이션,대만,


In [16]:
def explode_count(df: pd.DataFrame, col: str, version: str, top_n: int = 30) -> pd.DataFrame:
    counter = Counter()
    for value in df.loc[df['use_for_content_features'] == 1, col].fillna('').astype(str):
        for part in split_multi(value):
            counter[part] += 1
    return pd.DataFrame([
        {'version': version, 'field': col, 'value': k, 'count': v}
        for k, v in counter.most_common(top_n)
    ])

top_values = pd.concat([
    explode_count(unified_v2, 'unified_genres', 'v2'),
    explode_count(unified_v2, 'unified_countries', 'v2'),
    explode_count(unified_v2, 'unified_age_rating', 'v2'),
], ignore_index=True)
top_values.to_csv(TABLES_DIR / '03_top_unified_metadata_values_v2.csv', index=False, encoding='utf-8-sig')
top_values.head(40)

,version,field,value,count
0,v2,unified_genres,드라마,2099
1,v2,unified_genres,액션,1377
2,v2,unified_genres,스릴러,1234
3,v2,unified_genres,범죄,1234
4,v2,unified_genres,코미디,826
5,v2,unified_genres,SF,661
6,v2,unified_genres,판타지,661
7,v2,unified_genres,로맨스,655
8,v2,unified_genres,모험,487
9,v2,unified_genres,어드벤처,487


In [17]:
OUT_V1 = INTERIM_DIR / 'movie_metadata_unified_v1.csv'
OUT_V2 = INTERIM_DIR / 'movie_metadata_unified_v2.csv'
OUT_SUMMARY_JSON = INTERIM_DIR / 'movie_metadata_unified_v1_v2_summary.json'

unified_v1.to_csv(OUT_V1, index=False, encoding='utf-8-sig')
unified_v2.to_csv(OUT_V2, index=False, encoding='utf-8-sig')

summary = {
    'input_files': {
        'movie_master': str(PATH_MOVIE),
        'wavve': str(PATH_WAVVE),
        'kobis': str(PATH_KOBIS),
        'view_history': str(PATH_VIEW),
    },
    'output_files': {
        'movie_metadata_unified_v1': str(OUT_V1),
        'movie_metadata_unified_v2': str(OUT_V2),
        'summary_json': str(OUT_SUMMARY_JSON),
    },
    'input_rows': file_summary.to_dict('records'),
    'title_key_summary': key_summary.to_dict('records'),
    'coverage_summary': coverage.to_dict('records'),
    'quality_counts': quality_counts.to_dict('records'),
    'low_confidence_kobis_count': int(len(low_confidence_kobis)),
    'recommended_downstream_file': str(OUT_V2),
    'note': 'Use v2 for downstream content feature engineering. v1 is kept for audit and comparison.',
}
with open(OUT_SUMMARY_JSON, 'w', encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print('saved:', OUT_V1)
print('saved:', OUT_V2)
print('saved:', OUT_SUMMARY_JSON)

saved: /mnt/data/test_repo_03/park.ingyeom/_data/02_interim/movie_metadata_unified_v1.csv
saved: /mnt/data/test_repo_03/park.ingyeom/_data/02_interim/movie_metadata_unified_v2.csv
saved: /mnt/data/test_repo_03/park.ingyeom/_data/02_interim/movie_metadata_unified_v1_v2_summary.json


In [18]:
assert len(unified_v1) == len(movie), 'v1 row count must match Movie_Master row count.'
assert len(unified_v2) == len(movie), 'v2 row count must match Movie_Master row count.'
assert unified_v2['MOVIE_NUM'].isna().sum() == 0, 'MOVIE_NUM should not be missing.'
assert unified_v2['movie_title'].isna().sum() == 0, 'movie_title should not be missing.'

final_check = pd.DataFrame([
    {'check': 'v1_rows_equal_movie_master', 'value': len(unified_v1) == len(movie)},
    {'check': 'v2_rows_equal_movie_master', 'value': len(unified_v2) == len(movie)},
    {'check': 'v2_recommended_for_downstream', 'value': True},
    {'check': 'v2_low_confidence_rows_excluded_from_content_features', 'value': int((unified_v2['metadata_quality'].astype(str).str.startswith('D_') & (unified_v2['use_for_content_features'] == 1)).sum()) == 0},
])
final_check.to_csv(TABLES_DIR / '03_movie_metadata_final_checks.csv', index=False, encoding='utf-8-sig')
final_check

,check,value
0,v1_rows_equal_movie_master,True
1,v2_rows_equal_movie_master,True
2,v2_recommended_for_downstream,True
3,v2_low_confidence_rows_excluded_from_content_f...,True


## 03번 결론

이 노트북은 `Movie_Master_v1.csv`를 기준으로 Wavve와 KOBIS 메타데이터를 통합한다.

다운스트림 분석에서는 `movie_metadata_unified_v2.csv`를 사용한다.

`v2`는 Wavve 메타데이터를 우선 사용하고, Wavve가 없는 경우에만 KOBIS를 보완으로 사용한다. 단, KOBIS는 제목 유사도와 제목 내 연도 힌트로 검증하며, 저신뢰 매칭은 `use_for_content_features=0`으로 제외한다.

다음 단계인 `04_usage_feature_engineering.ipynb`는 02번에서 생성한 관측창 시청이력으로 유저별 사용 행동 피처를 만든다. 이후 `05_content_feature_engineering.ipynb`에서 이 노트북의 `movie_metadata_unified_v2.csv`를 사용한다.